> Pack Javi's synthetic data for training.

In [ ]:
import pathlib
import re

import scipy.io
import pandas as pd
import numpy as np

import torch

The directory containing the data

In [ ]:
data_directory = pathlib.Path.cwd().parent / 'javi' / 'simulations'
assert data_directory.exists()

An, e.g, FM sample file (though this will also be valid for AM and *carrier*)

In [ ]:
sample_file = next(data_directory.rglob('FM*.mat'))
sample_file

Metadata is extracted from the name of the file 

In [ ]:
re_signal = re.compile(r'(FM|AM|carrier)_SNR([0-9]*\.?[0-9]+)_Simul(\d+).*\.mat')
re_signal.match(sample_file.name).groups()

A noise sample file

In [ ]:
sample_file = next(data_directory.rglob('noise*.mat'))
sample_file

Metadata is extracted from the name of the file 

In [ ]:
re_noise = re.compile(r'noise_color([0-9]*\.?[0-9]+).*\.mat')
re_noise.match(sample_file.name).group(1)

A function to pack a bunch of `mat` files into *torch* `tensor`s

In [ ]:
def pack_dir(input_dir: str | pathlib.Path, output_file_stem: str | pathlib.Path) -> tuple[torch.Tensor, pd.DataFrame]:

    input_dir = pathlib.Path(input_dir)
    assert input_dir.exists()

    # for the sake of simplicity
    output_file_stem = pathlib.Path(output_file_stem)

    signal_files = list(input_dir.rglob('*.mat'))

    signals = []
    labels = []
    SNRs = []
    file_names = []
    colors = []


    for f in signal_files:

        signals.append(scipy.io.loadmat(f)['y'])

        if m := re_signal.match(f.name):
            
            # 3rd group is "id"
            label, SNR, _ = m.groups()
            
            color = np.nan
        elif m:=re_noise.match(f.name):
            label, SNR = 'noise', np.nan
            color = m.group(1)
        else:
            print(f'Could not match {f.name}')
            raise ValueError

        labels.append(label)
        SNRs.append(float(SNR))
        file_names.append(f.relative_to(input_dir))
        colors.append(color)

    # results are packed in torch tensors (rather than numpy arrays)
    signals = torch.from_numpy(np.stack(signals))
    torch.save(signals, output_file_stem.with_stem(output_file_stem.stem + '_signals').with_suffix('.pt'))

    df = pd.DataFrame({'label': labels, 'SNR': SNRs, 'color': colors, 'file name': file_names})
    df['label'] = df['label'].astype('category')
    df['file name'] = df['file name'].astype('string')

    df.to_parquet(output_file_stem.with_stem(output_file_stem.stem + '_labels').with_suffix('.parquet'))

    return signals, df


In [ ]:
signals, metadata_df = pack_dir(data_directory, '4_classes_modulated')
metadata_df.head()

In [ ]:
signals.shape

Minimum and maximum values are

In [ ]:
# max_value, min_value = signals.max(), signals.min()
# max_value, min_value

and in *absolute* value

In [ ]:
max_abs_value = signals.abs().max()
max_abs_value

Mean and standard devation:

In [ ]:
signals.mean(axis=0), signals.std(axis=0)

In [ ]:
# signals / max_abs_value